# Walker Lake

In this exercise we will see how to train a Gaussian Process model on the Walker Lake data.

Do not forget to enable the GPU in Colab.

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cmcrameri import cm

import geoml

import geoml.kernels as kr
import geoml.transform as tr
import geoml.warping as wp

In [ ]:
# constants
CMAP = cm.batlow
MAX_V = 1700.0

## The dataset

The dataset is included in the package. It contains a sample of 470 data points and a full 260x300 grid (the "exaustive" dataset as in the book).

In [ ]:
walker, walker_ex = geoml.datasets.walker()

print('Dataset:')
print(walker)
print("\nFull grid:")
print(walker_ex)

We will work with the $V$ variable.

Location map:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=[9, 9])
sc_v = ax.scatter(walker.coordinates[:, 0], walker.coordinates[:, 1],
                     c=walker.variables["V"].measurements.values, cmap=CMAP,
                     vmin=0, vmax=MAX_V)
ax.set_aspect("equal")
ax.set_title("Variable V")
ax.set_xlabel("X")
ax.set_ylabel("Y")
plt.colorbar(sc_v, ax=ax, aspect=40, shrink=0.9)

fig.show()

Histogram:

In [ ]:
fig, ax = plt.subplots(figsize=[9, 9])
ax.hist(walker.variables["V"].measurements.values,
        bins=30)
ax.set_xlabel("V")
ax.set_ylabel("Counts")
fig.show()

It can be seen that the data distribution is asymmetric and non-negative.

## Warped Gaussian process

We will use a warped model to deal with the characteristics of the data.

For the warping we scale the values, use the softplus function to keep the output positive and specify a monotonic spline for additional flexibility.

In [ ]:
# Warping (a.k.a normal score transform or anamorphosis)
warping_V = wp.ChainedWarping(
        wp.Scale(1, MAX_V),
        wp.Softplus(1),
        wp.ZScore(1),
        wp.Spline(1, knots_per_arm=2)  # higher = more flexibility
        )

# Choose a transform
# Isotropic
transf = tr.Isotropic(100)

# Anisotropic
# transf = tr.Anisotropy2D(
#     maxrange=100,
#     minrange_fct=0.5,
#     azimuth=135
# )

# Automatic anisotropic
# transf = tr.Anisotropy2DDynamic(
#     n_directions=9
# )

cov = kr.Covariance(
    kernel=kr.Spherical(),
    transform=transf
)


# The model
gp = geoml.models.GP(
    data=walker,
    variable="V",
    covariance=cov,
    warping=warping_V
)
gp.train(500)

In [ ]:
fig, ax = plt.subplots(figsize=[8, 6])
ax.plot(gp.training_log)
ax.set_xlabel("Iteration")
ax.set_ylabel("Log-likelihood")
fig.show()

In [ ]:
gp

### Prediction and results

The `walker_ex` object already has coordinates defined on a grid, so we use it to make the prediction.

In [ ]:
gp.predict(walker_ex)

In [ ]:
# The quantiles to plot are defined after the predictions.
# It is possible to compute any quantile corresponding to the
# [0, 1] probability interval.
walker_ex.variables['V'].reset_quantiles([0.025, 0.5, 0.975])

fig, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=[12, 10])

sc_v = ax[0, 0].imshow(walker_ex.variables["V"].quantiles[0.025].as_image(),
                       vmin=0, vmax=MAX_V, origin="lower", cmap=CMAP)
ax[0, 0].set_aspect("equal")
ax[0, 0].set_title("V - 2.5% quantile")

ax[0, 1].imshow(walker_ex.variables["V"].quantiles[0.5].as_image(),
                vmin=0, vmax=MAX_V, origin="lower", cmap=CMAP)
ax[0, 1].set_aspect("equal")
ax[0, 1].set_title("V - median")

ax[1, 0].imshow(walker_ex.variables["V"].quantiles[0.975].as_image(),
                vmin=0, vmax=MAX_V, origin="lower", cmap=CMAP)
ax[1, 0].set_aspect("equal")
ax[1, 0].set_title("V - 97.5% quantile")

ax[1, 1].imshow(walker_ex.variables["V"].measurements.as_image(),
                vmin=0, vmax=MAX_V, origin="lower", cmap=CMAP)
ax[1, 1].set_aspect("equal")
ax[1, 1].set_title("V - True values")

plt.colorbar(sc_v, ax=ax, shrink=0.95)

fig.show()

It can be seen that the predictions are non-negative. The asymmetry is harder to observe but can be inferred by the color scale. We can also visualize the warping, or normal score transform:

In [ ]:
normal = np.linspace(-4, 4, 1001)[:, None]
back_v = gp.warping.backward(normal)

fig, ax = plt.subplots(figsize=[8, 8])

ax.plot(normal, back_v, "-k")
ax.set_xlabel("Normal scores")
ax.set_ylabel("V - original values")

fig.show()